# 📝 통계 기초 과제 LV2(응용) — 기술통계·상관·추론

> 서로 다른 개념을 **두 개 이상 조합**하는 문제입니다. 여섯 나라의 의료비 지출·기대수명 데이터(`healthexp`)로 필터·기술통계·상관·중심극한정리·신뢰구간·p-value 를 이어서 연습합니다.

## 풀이 방법
1. 각 문제의 **답안 셀**(`# 여기에 코드를 작성하세요`)에 코드를 채웁니다.
2. 바로 아래 **자가채점 셀**(`# [자가채점]`)을 실행해 `✅ 통과!` 가 뜨면 성공이에요. (**그래프·서술 문제는 자가채점이 없어요** — 그래프는 완성 그림과 같은 모양으로 그리고, 서술은 관찰을 직접 적습니다.)
3. 막히면 `힌트` 를 펼쳐 보세요.

- 데이터는 `data/healthexp.csv` 를 씁니다(연도·국가·1인당 의료비 지출·기대수명).
- **문제마다 원본 CSV 를 다시 불러와** 시작하세요(앞 문제의 가공이 뒤 문제에 섞이지 않도록).
- 문제 **1·4·5·6·7·9·10·11** 에는 **서술 셀**이 있습니다 — 자가채점/그래프 아래 markdown 셀에 관찰을 직접 적으세요.

화이팅!

아래 셀을 먼저 실행해 분석 라이브러리와 한글 폰트를 준비하세요.

In [ ]:
# [제공 코드] 통계 분석에 쓸 라이브러리와 한글 폰트를 준비합니다.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지
sns.set_theme(font=KOREAN_FONT, rc={'axes.unicode_minus': False})

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을, `describe()` 로 수치 요약을 봅니다. (아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·수치 요약
df = pd.read_csv('data/healthexp.csv')
print('행·열 크기:', df.shape)
print('\n[앞 5행] head()'); display(df.head())
print('\n[열·자료형·결측] info()'); df.info()
print('\n[수치 요약] describe()'); display(df.describe())
print('\n[범주형 요약] describe(exclude="number")'); display(df.describe(exclude='number'))

## 1. 데이터를 살펴보고 관찰 적기 (서술)
**배경**: 분석을 시작하기 전, 위 `데이터 살펴보기` 결과를 바탕으로 이 데이터가 어떤 데이터인지 스스로 정리해 봅니다.

**요구사항**:
- 위 `# [제공 코드]`(head·info·describe) 실행 결과를 보고, 아래 **서술 셀**에 관찰을 **2~3문장**으로 적으세요.
- 다음을 담으면 좋아요: 몇 개 나라의 몇 개 행인지, 어떤 열이 있는지, 결측치가 있는지, 지출과 기대수명이 대략 어느 방향으로 함께 움직이는지.

**관찰 (서술)**

*(여기에 2~3문장으로 데이터 관찰을 적으세요 — 나라 수·행 수, 열 구성, 결측 여부, 지출과 기대수명의 방향)*

## 2. 미국의 지출 요약 통계 (필터 + 기술통계)
**배경**: 나라마다 의료비 지출 수준과 그 변동 폭이 다릅니다. 먼저 **미국(USA)** 한 나라만 뽑아 지출(`Spending_USD`)의 대표값과 산포를 정리합니다.

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- `Country` 가 `'USA'` 인 행만 골라 그 `Spending_USD` 의 **평균**을 `usa_mean`, **표준편차**를 `usa_std` 에 담으세요(표준편차는 `.std()` 기본값 = 표본표준편차 ddof=1).
- **변동계수**를 `usa_cv` 에 담으세요. 변동계수 = 표준편차 ÷ 평균 × 100 (백분율, %).
- 세 값 모두 소수 **둘째 자리**까지 비교합니다.

**예시**
```
round(usa_mean, 2) → 4388.57
round(usa_std, 2)  → 3386.31
round(usa_cv, 2)   → 77.16
```
<details><summary>힌트</summary>

```text
접근방법:
- 한 나라만 보려면 불리언 조건으로 그 나라 행만 거른다. 그 뒤 평균·표준편차를 구하고, 변동계수는 둘로 계산한다.

세부구현:
1. Country 가 미국인 행만 불리언 인덱싱으로 고른다
2. 그 Spending_USD 의 평균과 표준편차를 각각 구한다
3. 표준편차를 평균으로 나눈 뒤 100을 곱해 변동계수(백분율)를 만든다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]  통계값은 정확일치 대신 허용오차(abs<tol)로 비교합니다.
assert abs(usa_mean - 4388.57) < 0.01
assert abs(usa_std - 3386.31) < 0.01
assert abs(usa_cv - 77.16) < 0.01, '변동계수 = 표준편차 ÷ 평균 × 100 이에요'
print("✅ 문제2 통과!")

## 3. 가장 최근 연도의 지출 1위·최하위 나라와 격차 (연도 필터 + 최댓값·최솟값·격차)
**배경**: 가장 최근 연도에 어느 나라가 1인당 의료비를 가장 많이·가장 적게 썼는지 찾고, **1위와 최하위의 격차**가 얼마나 벌어져 있는지까지 계산해 나라 간 지출 불균형을 확인합니다.

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- 데이터에서 **가장 최근 연도**를 `latest_year` 에 담으세요(`Year` 의 최댓값).
- 그 연도의 행만 골라, `Spending_USD` 가 **가장 큰** 나라 이름을 `top_country`(문자열)·그 지출 값을 `top_spending`, **가장 작은** 나라 이름을 `bottom_country`(문자열)·그 지출 값을 `bottom_spending` 에 담으세요.
- 1위와 최하위의 **지출 격차**(`top_spending - bottom_spending`)를 `spending_gap` 에 담으세요.
- `top_spending`·`bottom_spending`·`spending_gap` 은 소수 **둘째 자리**까지 비교합니다.

**예시**
```
latest_year             → 2020
top_country             → 'USA'
round(top_spending, 2)  → 11859.18
bottom_country          → 'Japan'
round(bottom_spending, 2) → 4665.64
round(spending_gap, 2)  → 7193.54
```
<details><summary>힌트</summary>

```text
접근방법:
- 최근 연도는 Year 열의 최댓값이다. 그 연도만 거른 뒤, 지출이 가장 큰 행과 가장 작은 행의 나라·값을 각각 뽑는다.
- 격차는 1위 지출에서 최하위 지출을 빼면 된다.

세부구현:
1. Year 의 최댓값을 구해 latest_year 에 담는다
2. Year 가 latest_year 인 행만 불리언 인덱싱으로 고른다
3. 그 안에서 Spending_USD 가 최대인 행(idxmax)과 최소인 행(idxmin)을 찾아 나라·지출 값을 꺼낸다
4. 1위 지출에서 최하위 지출을 빼 격차를 구한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert latest_year == 2020
assert top_country == 'USA', '2020년 지출 1위는 미국이에요'
assert abs(top_spending - 11859.18) < 0.01
assert bottom_country == 'Japan', '2020년 지출 최하위는 일본이에요'
assert abs(bottom_spending - 4665.64) < 0.01
assert abs(spending_gap - 7193.54) < 0.01
print("✅ 문제3 통과!")

## 4. 지출과 기대수명 산점도 + 관찰 (관계 + 색 구분 + 해석)
**배경**: 1인당 의료비 지출(`Spending_USD`)이 클수록 기대수명(`Life_Expectancy`)이 어떻게 움직이는지, 나라(`Country`)별로 색을 달리해 그리고, **그림에서 무엇이 보이는지**까지 읽어 봅니다.

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- `Spending_USD`(x)와 `Life_Expectancy`(y)의 **산점도**(`sns.scatterplot`)를 그리되 `hue='Country'` 로 나라별 색을 나누세요. 결과 Axes 를 `ax` 에 저장하고 제목을 `set_title` 으로 다세요.
- 그래프는 자가채점이 없습니다. 그린 뒤 아래 **서술 셀**에, 점들이 **어느 방향**으로 늘어서는지와 **오른쪽 위(고지출·고수명)** 에 주로 어느 나라가 있는지를 **1~2문장**으로 적으세요.

**예시**
```
x축: Spending_USD,  y축: Life_Expectancy
```
<details><summary>힌트</summary>

```text
접근방법:
- 두 수치 변수의 관계는 산점도로 본다. 점이 어느 방향으로 늘어서는지 눈으로 읽는다.
- 세 번째 정보(나라)는 hue 로 색을 나눠 겹쳐 본다.

세부구현:
1. healthexp 를 다시 불러온다
2. scatterplot 으로 x 는 Spending_USD, y 는 Life_Expectancy 를 찍되 hue 에 Country 를 준다
3. 결과 Axes 를 ax 에 담고 제목을 단다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv2_q4.png" width="560"/>

In [ ]:
# 여기에 코드를 작성하세요

**관찰 (서술)**

*(여기에 1~2문장으로 서술하세요 — 점들이 늘어서는 방향, 오른쪽 위·왼쪽 위에 눈에 띄는 나라)*

## 5. 지출과 기대수명의 상관계수 (피어슨 + 해석)
**배경**: 산점도에서 눈으로 본 관계를 숫자 하나로 요약합니다. 피어슨 상관계수로 두 변수가 **직선으로** 얼마나 함께 움직이는지 잽니다.

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- `stats.pearsonr` 로 `Spending_USD` 와 `Life_Expectancy` 의 상관계수를 구해 `pearson_r` 에 담으세요(반환값의 **첫 번째**가 상관계수입니다).
- 상관계수 절댓값을 기준으로 강도를 판단해 `pearson_strength` 에 담으세요: 0.3 미만이면 `'약함'`, 0.3 이상 0.7 미만이면 `'중간'`, 0.7 이상이면 `'강함'`.
- `pearson_r` 은 소수 **셋째 자리**까지 비교합니다.
- 자가채점 아래 **서술 셀**에, 지출과 기대수명이 어느 방향으로 얼마나 강하게 이어지는지 **1문장 이상** 적으세요.

**예시**
```
round(pearson_r, 3) → 0.579
pearson_strength    → '중간'
```
<details><summary>힌트</summary>

```text
접근방법:
- pearsonr 는 두 값을 돌려주므로 첫 번째(상관계수)만 쓴다.
- 강도는 절댓값을 세 구간(약함·중간·강함)으로 나눠 판단한다.

세부구현:
1. pearsonr 에 두 열을 넣어 첫 번째 반환값을 pearson_r 에 담는다
2. pearson_r 의 절댓값을 0.3·0.7 을 경계로 조건 분기해 약함/중간/강함 문자열을 정한다
3. 그 문자열을 pearson_strength 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(pearson_r - 0.579) < 0.01
assert pearson_strength == '중간', '0.3 이상 0.7 미만이면 중간이에요'
print("✅ 문제5 통과!")

**관찰 (서술)**

*(여기에 1문장 이상으로 서술하세요 — 지출과 기대수명의 방향과 상관 강도)*

## 6. 순위로 본 상관 (스피어만 + 비교)
**배경**: 피어슨은 직선 관계를 재지만, 관계가 살짝 휘어 있으면 순위 기반의 스피어만 상관이 더 잘 잡아냅니다. 두 값을 비교합니다.

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- `stats.spearmanr` 로 `Spending_USD` 와 `Life_Expectancy` 의 스피어만 상관계수를 구해 `spearman_rs` 에 담으세요(반환값의 **첫 번째**).
- `spearman_rs` 는 소수 **셋째 자리**까지 비교합니다.
- 자가채점 아래 **서술 셀**에, 스피어만 상관계수를 앞 문제의 피어슨 상관계수와 비교해 어느 쪽이 더 큰지와 그 의미를 **1문장 이상** 적으세요.

**예시**
```
round(spearman_rs, 3) → 0.747
```
<details><summary>힌트</summary>

```text
접근방법:
- spearmanr 도 두 값을 돌려주므로 첫 번째(상관계수)만 쓴다.
- 앞 문제의 피어슨 값과 크기를 비교한다.

세부구현:
1. spearmanr 에 두 열을 넣어 첫 번째 반환값을 spearman_rs 에 담는다
2. 소수 셋째 자리로 반올림해 확인한다
3. 피어슨(0.579)과 크기를 견줘 서술한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(spearman_rs - 0.747) < 0.01
print("✅ 문제6 통과!")

**관찰 (서술)**

*(여기에 1문장 이상으로 서술하세요 — 스피어만과 피어슨의 크기 비교와 그 의미)*

## 7. 표본평균의 분포 — 중심극한정리 (시뮬레이션 + 관찰)
**배경**: 원래 지출(`Spending_USD`) 분포는 한쪽으로 치우쳐 있습니다. 그런데 이 모집단에서 30개씩 뽑아 **평균**을 여러 번 구하면, 그 평균들의 분포는 어떤 모양이 될까요? 중심극한정리를 눈으로 확인합니다.

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- `Spending_USD` 전체를 모집단으로 삼아, 크기 **30**인 표본을 **복원추출**해 그 평균을 구하는 일을 **2000번** 반복하고, 그 표본평균들을 `sample_means` 에 모으세요.
- 재현을 위해 맨 앞에서 `rng = np.random.default_rng(42)` 로 난수 생성기를 만들어 `rng.choice(모집단, size=30, replace=True)` 로 표본을 뽑으세요.
- `sample_means` 의 **히스토그램**을 `sns.histplot(sample_means, bins=30, kde=True)` 로 그려 결과 Axes 를 `ax` 에 저장하고 제목을 다세요.
- 그래프 자체는 채점하지 않지만, `sample_means` 는 자가채점합니다. 아래 **서술 셀**에도 원래 지출 분포의 모양과 표본평균 분포의 모양이 어떻게 다른지 **2문장 이상** 관찰을 적으세요.

**예시**
```
len(sample_means) → 2000
x축: 표본평균,  종 모양(정규분포)에 가까운 분포
```
<details><summary>힌트</summary>

```text
접근방법:
- 모집단에서 표본을 뽑아 평균을 구하는 일을 아주 여러 번 반복하고, 그 평균들을 모아 히스토그램으로 그린다.
- 복원추출은 rng.choice 에 replace=True 로 준다.

세부구현:
1. Spending_USD 값 전체를 모집단 배열로 꺼낸다
2. 반복문(또는 리스트 컴프리헨션)으로 2000번, 매번 크기 30 표본을 복원추출해 평균을 구해 모은다
3. 모은 표본평균들을 histplot 으로 그리고 ax 에 담아 제목을 단다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv2_q7.png" width="560"/>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]  그래프는 채점하지 않고, 모은 표본평균 2000개만 확인합니다.
assert len(sample_means) == 2000
assert abs(np.mean(sample_means) - 2789.34) < 60, '표본평균들의 중심은 모평균 부근이어야 합니다'
assert 250 < np.std(sample_means) < 550, '표본평균들의 흩어짐이 이론값(σ/√30 ≈ 401)과 크게 어긋납니다'
print("✅ 문제7 통과!")

**관찰 (서술)**

*(여기에 2문장 이상으로 서술하세요 — 원래 지출 분포의 모양 vs 표본평균 분포의 모양)*

## 8. 기대수명 평균의 95% 신뢰구간 (공식)
**배경**: 표본에서 구한 평균 하나만으로는 참 평균을 알 수 없습니다. 평균이 어느 범위 안에 있을지 **95% 신뢰구간**으로 나타냅니다.

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- `Life_Expectancy` 전체 평균에 대한 **95% 신뢰구간**을 구해 하한을 `ci_low`, 상한을 `ci_high` 에 담으세요.
- 신뢰구간은 **표본평균을 중심으로 표준오차만큼 좌우로 벌린 범위**입니다. `scipy.stats` 에서 표준오차를 구하는 함수(`sem`)와, 신뢰수준·평균(`loc`)·표준오차(`scale`)를 받아 (하한, 상한)을 돌려주는 정규분포 구간 함수(`norm.interval`)를 쓰세요.
- **주의**: `scale` 에 표준편차가 아니라 **표준오차**를 넣어야 *평균*의 신뢰구간이 됩니다.
- 두 값 모두 소수 **둘째 자리**까지 비교합니다.
- **주의**: 이 데이터는 국가·연도가 섞여 있어 완전히 독립 표본은 아니지만, 이 문제에서는 독립으로 가정하고 신뢰구간 공식을 연습합니다(실제로는 구간이 더 넓어집니다).

**예시**
```
round(ci_low, 2)  → 77.52
round(ci_high, 2) → 78.3
```
<details><summary>힌트</summary>

```text
접근방법:
- 신뢰구간은 평균을 중심으로 표준오차만큼 좌우로 벌린 범위다.
- norm.interval 에 신뢰수준·평균(loc)·표준오차(scale)를 넣으면 (하한, 상한)을 돌려준다.

세부구현:
1. Life_Expectancy 의 평균과 표준오차(stats.sem)를 구한다
2. 정규분포 구간 함수에 신뢰수준 0.95 와 평균·표준오차를 넣어 호출한다
3. 돌려받은 두 값을 각각 ci_low, ci_high 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(ci_low - 77.52) < 0.01
assert abs(ci_high - 78.3) < 0.01
print("✅ 문제8 통과!")

## 9. 두 주장을 p-value 로 견주기 (+ 신뢰구간과의 관계)
**배경**: 신뢰구간은 어떤 주장이 **안이냐 밖이냐**만 답합니다. 그런데 밖에 있는 주장들끼리도 **얼마나** 어긋났는지는 서로 다르죠. 그 어긋난 **정도**를 숫자 하나로 재는 것이 **p-value** 입니다. 두 보고서의 주장을 p-value 로 견주고, 그 결과가 문제 8 의 신뢰구간과 **일치하는지** 확인합니다.

- 보고서 A: "이 나라들의 평균 기대수명은 **78.5세** 다"
- 보고서 B: "아니다, **78.0세** 다"

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- `Life_Expectancy` 의 표본평균과 표준오차(`stats.sem`)로, 각 주장에 대한 **t 통계량**을 `t = (표본평균 − 주장값) / 표준오차` 로 구하세요.
- **양측 p-value** 를 `2 * stats.t.sf(abs(t), df=n-1)` 로 구해 각각 `p_a`(주장 78.5)·`p_b`(주장 78.0) 에 담으세요(`n` 은 값의 개수, 양측이라 **2를 곱합니다**).
- 두 p 값 모두 소수 **넷째 자리**까지 비교합니다.
- 각 주장이 문제 8 의 95% 신뢰구간 `[77.52, 78.30]` 안에 있는지 판단해, 문자열 `'구간 밖'` 또는 `'구간 안'` 을 `verdict_a`·`verdict_b` 에 담으세요.
- 자가채점 아래 **서술 셀**에, **p < 0.05 인 주장과 신뢰구간 밖인 주장이 서로 일치하는지**를 **1~2문장**으로 적으세요.

**예시**
```
round(p_a, 4) → 0.0031     verdict_a → '구간 밖'
round(p_b, 4) → 0.6478     verdict_b → '구간 안'
```
<details><summary>힌트</summary>

```text
접근방법:
- 주장값에서 표본평균이 표준오차 몇 칸만큼 떨어졌는지가 t 통계량이다.
- 그 t 가 우연히 나올 확률을 t분포의 꼬리 넓이로 재고, 양쪽 꼬리를 보므로 2배 한다.

세부구현:
1. Life_Expectancy 의 평균·표준오차(stats.sem)·개수를 구한다
2. 주장값마다 t 를 계산하고 stats.t.sf 에 abs(t) 와 자유도를 넣어 한쪽 꼬리 넓이를 구한 뒤 2배 한다
3. 주장값이 [77.52, 78.30] 사이에 있는지 비교해 판정 문자열을 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(p_a - 0.0031) < 0.0005
assert abs(p_b - 0.6478) < 0.0005
assert verdict_a == '구간 밖'
assert verdict_b == '구간 안'
assert (p_a < 0.05) == (verdict_a == '구간 밖')   # 구간 밖 ⟺ p < 0.05
assert (p_b < 0.05) == (verdict_b == '구간 밖')
print("✅ 문제9 통과!")

**관찰 (서술)**

*(여기에 1~2문장으로 서술하세요 — p < 0.05 인 주장과 신뢰구간 밖인 주장이 서로 일치하나요?)*

## 10. 세 변수 상관행렬 히트맵 + 관찰 (상관 + 색칠 + 해석)
**배경**: 연도·지출·기대수명 세 수치 변수가 서로 얼마나 함께 움직이는지 **상관행렬**로 구하고, 히트맵으로 한눈에 본 뒤, **어느 두 변수의 관계가 가장 강한지**까지 읽어 봅니다.

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- `Year`·`Spending_USD`·`Life_Expectancy` 세 열의 **상관행렬**을 `df[[...]].corr()` 로 구해 `corr` 에 담으세요(shape `(3, 3)`).
- `corr` 를 `sns.heatmap(corr, annot=True, cmap='Blues')` 로 그려 숫자를 칸에 표시하세요. 결과 Axes 를 `ax` 에 저장하고 제목을 다세요.
- 그래프는 자가채점이 없습니다. 그린 뒤 아래 **서술 셀**에, 대각선(자기 자신, 값 1)을 빼고 **가장 큰 상관을 보인 두 변수의 짝**과 그 값을 **1~2문장**으로 적으세요.

**예시**
```
corr.shape → (3, 3)
x·y축: Year, Spending_USD, Life_Expectancy
```
<details><summary>힌트</summary>

```text
접근방법:
- 여러 수치 열의 상관을 한 번에 보려면 corr 로 상관행렬을 만들고, 그 표를 heatmap 으로 색칠한다.

세부구현:
1. 세 열만 뽑아 corr 로 상관행렬 corr 를 만든다
2. heatmap 으로 corr 를 그리되 annot=True 로 칸마다 숫자를 적는다
3. 결과 Axes 를 ax 에 담고 제목을 단다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv2_q10.png" width="560"/>

In [ ]:
# 여기에 코드를 작성하세요

**관찰 (서술)**

*(여기에 1~2문장으로 서술하세요 — 대각선을 뺀 가장 강한 상관의 두 변수 짝과 그 값)*

## 11. 인사이트 리포트 (종합 서술)
**배경**: 지금까지 구한 상관계수와 신뢰구간을 근거로, "의료비 지출과 기대수명은 어떤 관계인가"를 한 문단으로 정리합니다.

**요구사항**:
- 앞 문제들에서 구한 **상관계수(피어슨·스피어만)** 와 **기대수명 평균의 95% 신뢰구간**을 근거로 들며, 지출과 기대수명의 관계를 **3문장 이상**으로 아래 서술 셀에 적으세요.
- 상관은 인과가 아니라는 점(지출이 많다고 반드시 더 오래 사는 것은 아님)도 한 문장 포함하면 좋습니다.

**인사이트 리포트 (서술)**

*(여기에 3문장 이상으로 서술하세요 — 상관계수·신뢰구간을 근거로 지출과 기대수명의 관계, 그리고 상관≠인과)*